# Pharmacon — Automated Dynamic Multi-Writer Cohort Pipeline

This notebook dynamically handles any number of writers ($N = 2$ up to $30+$) without requiring any code changes:
1. **Auto-Discovery:** Automatically scans `/content/dataset/` for writer subdirectories (`W01`, `W02`, ...).
2. **Dynamic Sheet Mapping:** Detects Train (`TS1-4`), Validation (`VS`), and Held-Out (`HS1-2`) sheets per writer.
3. **Multi-Writer Pooling:** Pools all $16 \times N$ enrollment lines for training and $4 \times N$ lines for validation.
4. **Generic Baseline vs Adapted LoRA:** Evaluates both models on strictly sealed $8 \times N$ held-out lines.
5. **Cohort Benchmark Report:** Outputs individual writer deltas and overall Macro CER/WER.
6. **Clinical Entity Resolution:** Passes held-out predictions through FormularyMatcher (NLEM 2022).


In [ ]:
# Cell 1: Install dependencies and unpack cohort data
!pip install -q "sentencepiece>=0.2.0" "protobuf>=3.20.3" "transformers>=4.40.0" "peft>=0.10.0" "accelerate>=0.28.0" "pymupdf>=1.24.0" "rapidfuzz>=3.0.0"
!unzip -q -o /content/pharmacon_cohort_data.zip -d /content/
print("Environment initialized and cohort package unpacked successfully.")


In [ ]:
# Cell 2: Automated Multi-Writer Cohort Training & Evaluation Engine
import os, sys, copy, cv2, torch
import numpy as np
from transformers import AutoImageProcessor, XLMRobertaTokenizer, TrOCRProcessor, VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model
from pharmacon_ai import process_photo, evaluate_condition, print_comparison
from pharmacon_ai.formulary import FormularyMatcher

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

DATASET_DIR = "/content/dataset"
PDF_PATH    = "/content/Pharmacon_Writer_Calibration_Protocol.pdf"

# --- 1. DYNAMIC WRITER DISCOVERY & SHEET MAPPING ---
def map_writer_sheets(writer_dir):
    files = os.listdir(writer_dir)
    mapping = {}
    for f in sorted(files):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        fl = f.lower()
        full = os.path.join(writer_dir, f)
        if "ts1" in fl or "trains1" in fl or ("train" in fl and "1" in fl):
            mapping[2] = full
        elif "ts2" in fl or "trains2" in fl or ("train" in fl and "2" in fl):
            mapping[3] = full
        elif "ts3" in fl or "trains3" in fl or ("train" in fl and "3" in fl):
            mapping[4] = full
        elif "ts4" in fl or "trains4" in fl or ("train" in fl and "4" in fl):
            mapping[5] = full
        elif "vs" in fl or "valid" in fl:
            mapping[6] = full
        elif "hs1" in fl or "heldouts1" in fl or ("held" in fl and "1" in fl):
            mapping[7] = full
        elif "hs2" in fl or "heldouts2" in fl or ("held" in fl and "2" in fl):
            mapping[8] = full
    return mapping

writers = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f"\nDetected {len(writers)} writers in cohort: {writers}")

# --- 2. CROP AND POOL DATA ACROSS ALL WRITERS ---
train_crops, train_labels = [], []
val_crops, val_labels = [], []
test_data = {}  # {writer_id: {'crops': {lid: img}, 'labels': {lid: text}}}

train_ids = [f"E{i:02d}" for i in range(1, 17)]
val_ids   = [f"V{i:02d}" for i in range(1, 5)]
test_ids  = [f"T{i:02d}" for i in range(1, 9)]

for w_id in writers:
    w_dir = os.path.join(DATASET_DIR, w_id)
    mapping = map_writer_sheets(w_dir)
    print(f"  [{w_id}] Mapped {len(mapping)}/7 pages -> Extracting crops...")
    w_lines = {}
    for page_idx, p_path in sorted(mapping.items()):
        w_lines.update(process_photo(p_path, PDF_PATH, page_idx, writer_id=w_id))
    
    # Accumulate training lines
    for lid in train_ids:
        if lid in w_lines:
            train_crops.append(cv2.cvtColor(w_lines[lid]["image"], cv2.COLOR_BGR2RGB))
            train_labels.append(w_lines[lid]["label"])
            
    # Accumulate validation lines
    for lid in val_ids:
        if lid in w_lines:
            val_crops.append(cv2.cvtColor(w_lines[lid]["image"], cv2.COLOR_BGR2RGB))
            val_labels.append(w_lines[lid]["label"])
            
    # Isolated sealed test lines per writer
    test_data[w_id] = {
        "crops": {lid: cv2.cvtColor(w_lines[lid]["image"], cv2.COLOR_BGR2RGB) for lid in test_ids if lid in w_lines},
        "labels": {lid: w_lines[lid]["label"] for lid in test_ids if lid in w_lines}
    }

print(f"\nTotal Pooled Training Lines   : {len(train_crops)} ({len(writers)} writers x 16 lines)")
print(f"Total Pooled Validation Lines : {len(val_crops)} ({len(writers)} writers x 4 lines)")
print(f"Total Sealed Test Lines       : {sum(len(v['labels']) for v in test_data.values())} ({len(writers)} writers x 8 lines)")

# --- 3. LOAD BASELINE MODEL ---
MODEL_ID = "microsoft/trocr-small-handwritten"
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
tokenizer       = XLMRobertaTokenizer.from_pretrained(MODEL_ID)
processor       = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
model.config.decoder_start_token_id = processor.tokenizer.eos_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# --- 4. EVALUATE GENERIC BASELINE ACROSS COHORT ---
print("\n--- Evaluating Generic Baseline across all writers ---")
model.eval()
generic_results_per_writer = {}
with torch.no_grad():
    for w_id in writers:
        w_preds = {}
        for lid, img in test_data[w_id]["crops"].items():
            pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
            ids = model.generate(pv, max_new_tokens=64, num_beams=4, early_stopping=True, no_repeat_ngram_size=3, repetition_penalty=1.3)
            w_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        res = evaluate_condition(w_preds, test_data[w_id]["labels"])
        generic_results_per_writer[w_id] = res
        print(f"  [{w_id}] Generic Baseline: CER = {res['macro']['cer']:.4f} | WER = {res['macro']['wer']:.4f}")

# --- 5. CONFIGURE LORA ADAPTER ---
try:
    import peft.import_utils; peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao; peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

targets = [
    "encoder.layers.10.attention.q_proj", "encoder.layers.10.attention.v_proj",
    "encoder.layers.11.attention.q_proj", "encoder.layers.11.attention.v_proj"
]
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.15, target_modules=targets, bias="none")
peft_model = get_peft_model(model, lora_config)

# --- 6. TRAIN POOLED LORA ADAPTER ---
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)
best_val_loss, best_weights, patience, p_count = float("inf"), None, 5, 0

print(f"\n--- Training Multi-Writer Pooled LoRA ({len(train_crops)} lines, Early Stopping on {len(val_crops)} lines) ---")
for epoch in range(1, 41):
    peft_model.train()
    t_loss = 0.0
    for img, text in zip(train_crops, train_labels):
        pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
        lbl = processor.tokenizer(text=text, return_tensors="pt").input_ids.to(device)
        loss = peft_model(pixel_values=pv, labels=lbl).loss
        opt.zero_grad(); loss.backward(); opt.step()
        t_loss += loss.item()
    t_loss /= len(train_crops)

    peft_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for img, text in zip(val_crops, val_labels):
            pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
            lbl = processor.tokenizer(text=text, return_tensors="pt").input_ids.to(device)
            v_loss += peft_model(pixel_values=pv, labels=lbl).loss.item()
    v_loss /= len(val_crops)

    if epoch % 5 == 0 or v_loss < best_val_loss:
        print(f"Epoch {epoch:02d} | Train: {t_loss:.4f} | Val: {v_loss:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = copy.deepcopy(peft_model.state_dict())
        p_count = 0
    else:
        p_count += 1
        if p_count >= patience:
            print(f"Early stopping at epoch {epoch}. Restoring best checkpoint (Val Loss: {best_val_loss:.4f}).")
            break

if best_weights:
    peft_model.load_state_dict(best_weights)

# --- 7. EVALUATE ADAPTED MODEL ACROSS COHORT ---
print("\n--- Evaluating Adapted Model across all writers ---")
peft_model.eval()
adapted_results_per_writer = {}
adapted_preds_all = {}
with torch.no_grad():
    for w_id in writers:
        w_preds = {}
        for lid, img in test_data[w_id]["crops"].items():
            pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
            ids = peft_model.generate(pv, max_new_tokens=64, num_beams=4, early_stopping=True, no_repeat_ngram_size=3, repetition_penalty=1.3)
            w_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        res = evaluate_condition(w_preds, test_data[w_id]["labels"])
        adapted_results_per_writer[w_id] = res
        adapted_preds_all[w_id] = w_preds

# --- 8. COHORT SUMMARY BENCHMARK TABLE ---
print("\n" + "="*90)
print(f"PHARMACON MULTI-WRITER COHORT BENCHMARK REPORT ({len(writers)} WRITERS POOLED)")
print("="*90)
print(f"{'Writer':<10} {'Generic CER':<14} {'Adapted CER':<14} {'Δ CER':<12} {'Generic WER':<14} {'Adapted WER':<14} {'Δ WER':<12}")
print("-"*90)

g_cers, a_cers, g_wers, a_wers = [], [], [], []
for w_id in writers:
    g_c = generic_results_per_writer[w_id]['macro']['cer']
    a_c = adapted_results_per_writer[w_id]['macro']['cer']
    g_w = generic_results_per_writer[w_id]['macro']['wer']
    a_w = adapted_results_per_writer[w_id]['macro']['wer']
    g_cers.append(g_c); a_cers.append(a_c); g_wers.append(g_w); a_wers.append(a_w)
    d_c = a_c - g_c
    d_w = a_w - g_w
    print(f"{w_id:<10} {g_c:<14.4f} {a_c:<14.4f} {d_c:<+12.4f} {g_w:<14.4f} {a_w:<14.4f} {d_w:<+12.4f}")

print("-"*90)
cohort_g_cer = np.mean(g_cers)
cohort_a_cer = np.mean(a_cers)
cohort_g_wer = np.mean(g_wers)
cohort_a_wer = np.mean(a_wers)
print(f"{'COHORT':<10} {cohort_g_cer:<14.4f} {cohort_a_cer:<14.4f} {cohort_a_cer-cohort_g_cer:<+12.4f} {cohort_g_wer:<14.4f} {cohort_a_wer:<14.4f} {cohort_a_wer-cohort_g_wer:<+12.4f}")
print("="*90)
print(f">> Pooled Cohort Macro CER: {cohort_g_cer:.4f} → {cohort_a_cer:.4f} ({(cohort_a_cer - cohort_g_cer)/cohort_g_cer * 100:+.2f}%)")
print(f">> Pooled Cohort Macro WER: {cohort_g_wer:.4f} → {cohort_a_wer:.4f} ({(cohort_a_wer - cohort_g_wer)/cohort_g_wer * 100:+.2f}%)")

# --- 9. CLINICAL RESOLUTION ACROSS COHORT ---
print("\n" + "="*90)
print("CLINICAL RESOLUTION (Formulary Matching across Held-Out Lines)")
print("="*90)
matcher = FormularyMatcher()
for w_id in writers:
    print(f"\n--- Writer {w_id} Results ---")
    for lid in sorted(test_data[w_id]['labels'].keys()):
        raw = adapted_preds_all[w_id][lid]
        target = test_data[w_id]['labels'][lid]
        res = matcher.resolve(raw)
        top = res["top_candidates"][0] if res["top_candidates"] else {}
        ent = res["parsed_entities"]
        drug = top.get("generic_name", "UNKNOWN")
        conf = top.get("confidence", 0.0)
        gate = res["safety_gate"]["status"]
        strength = ent.get("strength") or ""
        form = ent.get("form") or ""
        resolved_str = f"{drug} {strength} {form}".strip()
        print(f"[{lid}] Target   : {target}")
        print(f"     Raw OCR  : {raw}")
        print(f"     Resolved : {resolved_str} (Conf: {conf:.2f}, Gate: {gate})")
